# Class 4 lab: features and choosing a model

**CML3113, Mining and Petroleum Engineering**  |  Week 11  |  KLAi at IUEA

In the lab you test a new feature with a feature ablation, meet two new models (ridge regression and k-nearest neighbours), build one model comparison table of five candidate models that predict **crusher energy demand** (in **kWh/t**) and start your decision memo.

**How to use this notebook**

1. Run the cells in order from the top. Press Shift and Enter to run a cell.
2. Every step has the same rhythm: **PREDICT** what will happen, **RUN** the cell, **READ** the output, then **CHANGE** one thing and see what moves.
3. Type your answers in the lines that start with `>`. Double-click a text cell to edit it.
4. If you get an error, read the last line first. The slide called "Four errors you will meet" lists the common ones.

**The words** (also on the slides): *feature engineering* means building a new feature (an input variable) from features you already have. A *feature ablation* (also called an *ablation study*) is a fair test of one change: train the model with the feature and without it, on the same rows, and compare the two errors. *Ridge regression* is linear regression with *regularisation*: an L2 penalty on large weights (coefficients). *Alpha* is the regularisation strength. *k-NN* means k-nearest neighbours: to predict a new row, average the targets of the k training rows closest to it. k is a *hyperparameter*, a setting you choose before training. *Model selection* means choosing one model from several candidate models. *Parsimony* means preferring the simpler model when scores are close. A *decision memo* is a one-page note with your choice, your numbers and your limits. A related industry term is *model card*.

**Two metrics.** *MAE* is the mean absolute error: the average size of the prediction error, in the same unit as the target. *RMSE* is the root mean squared error, which counts large errors more.

---
## Step 1: Prepare the data again

This notebook stands alone. Upload `engineering_data.csv` in the Files panel of Colab if you have not already. The cell repeats the Class 1 preparation: it cleans a copy of the table and splits the rows by equipment group into a training set (E1 to E4), a validation set (E5) and a held-out test set (E6, the hold-out set).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

raw_df = pd.read_csv("engineering_data.csv")                  # the raw table is never changed
clean_df = raw_df.drop_duplicates("record_id").copy()          # keep the first copy of each record_id
has_target = clean_df["target"].notna()                        # True where the target variable is recorded
load_ok = clean_df["load_pct"].between(0, 100)                 # True where load is between 0 and 100
clean_df = clean_df[has_target & load_ok].copy()
print("rows left to model with:", len(clean_df))

# E1 to E4 form the training set, E5 the validation set, E6 the held-out test set
train_df = clean_df[clean_df["equipment_id"].isin(["E1", "E2", "E3", "E4"])].copy()
valid_df = clean_df[clean_df["equipment_id"] == "E5"].copy()
test_df = clean_df[clean_df["equipment_id"] == "E6"].copy()   # held-out test set: we do not touch it today

feature_columns = ["load_pct", "ambient_c"]
X_train, y_train = train_df[feature_columns], train_df["target"]
X_valid, y_valid = valid_df[feature_columns], valid_df["target"]
print("training, validation, test rows:", len(train_df), len(valid_df), len(test_df))

**READ the output.** Do the numbers match the "Check your numbers" slide? If not, stop and ask before you go on.

> My answer: 

---
## Step 2: One loop, three models

A dictionary holds each model under its name, and a loop gives every model the same treatment. `with_imputer` is a helper, which means a small function that we write once and reuse. It builds a scikit-learn pipeline (a Pipeline chains preprocessing and a model): median imputation (`SimpleImputer`) fills each missing value (NaN, an empty cell) with the median of that column in the training set, and then the model is trained (fitted). Every model is evaluated on the validation set with the same metric, the mean absolute error (MAE), in kWh/t. The baseline model (`DummyRegressor`) always predicts the mean of the training target.

**PREDICT first.** Which of the three will have the smallest error: the baseline model, linear regression or the decision tree?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

def with_imputer(model):
    # first fill missing values with the median of the training set (imputation), then train the model
    return make_pipeline(SimpleImputer(strategy="median"), model)

models = {
    "Baseline model": with_imputer(DummyRegressor()),
    "Linear regression": with_imputer(LinearRegression()),
    "Decision tree, depth 4": with_imputer(DecisionTreeRegressor(max_depth=4, random_state=17)),   # same seed every time
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)          # training (fitting) the model
    y_pred = model.predict(X_valid)      # inference: predictions for the validation set
    mae = mean_absolute_error(y_valid, y_pred)
    rmse = mean_squared_error(y_valid, y_pred) ** 0.5
    rows.append([name, mae, rmse])
results = pd.DataFrame(rows, columns=["model", "MAE", "RMSE"]).round(2)
print(results.to_string(index=False))

**READ the output.** Do the three rows match the Class 1 model comparison table? Why does the loop keep the comparison fair?

> My answer: 

---
## Step 3: A new feature and a feature ablation

Idea from engineering sense: a crusher working hard in hot air may need more than load and temperature explain separately. We build `load_x_ambient` (load_pct times ambient_c, an interaction feature, which is a product of two features) and also `load_squared` (load_pct times itself). A feature ablation (ablation study) then changes one thing at a time: each version of the linear regression model is trained on the same training set and scored on the same validation set.

**PREDICT first.** Will the new feature lower the error by a lot, a little or not at all? Will removing `ambient_c` raise it by a lot or a little?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# build each new feature on BOTH tables, with the same rule
for table in (train_df, valid_df):
    table["load_x_ambient"] = table["load_pct"] * table["ambient_c"]
    table["load_squared"] = table["load_pct"] ** 2

feature_sets = {
    "Remove ambient_c (load only)": ["load_pct"],
    "Two features (start)": ["load_pct", "ambient_c"],
    "Add load_x_ambient": ["load_pct", "ambient_c", "load_x_ambient"],
    "Add load_squared": ["load_pct", "ambient_c", "load_squared"],
}

for name, columns in feature_sets.items():
    model = with_imputer(LinearRegression())
    model.fit(train_df[columns], y_train)
    mae = mean_absolute_error(y_valid, model.predict(valid_df[columns]))
    print(f"{name}: MAE {mae:.2f} kWh/t")

**READ the output.** Which change moved the error a lot, and which hardly at all? Would you keep `load_x_ambient`? Give the reason in one sentence.

> My answer: 

**The learned parameters.** Each linear regression model is a formula: prediction = intercept (bias) + weight 1 times feature 1 + weight 2 times feature 2 (and weight 3 times feature 3 for the extended model). The weights are also called coefficients. One more unit of a feature adds its weight to the prediction, when the other features stay fixed. The next cell prints these numbers for model A (multiple linear regression on two features) and model B (plus the interaction feature `load_x_ambient`). They are the numbers on the formula slide.

In [ ]:
for name, columns in [("Model A, two features", ["load_pct", "ambient_c"]), ("Model B, plus load_x_ambient", ["load_pct", "ambient_c", "load_x_ambient"])]:
    pipeline = with_imputer(LinearRegression())
    pipeline.fit(train_df[columns], y_train)
    linear_model = pipeline.steps[-1][1]        # the linear regression model itself, without the imputation step
    weights = ", ".join(f"weight for {c} {w:.4f}" for c, w in zip(columns, linear_model.coef_))
    print(f"{name}: intercept {linear_model.intercept_:.4f}, {weights}")

**READ the output.** Which weight is the smallest in model B? What does a tiny weight on `load_x_ambient` tell you about the new feature?

> My answer: 

**CHANGE one thing.** Make your own feature, for example `ambient_squared`, on both tables and add it to `feature_sets`. Run the loop again. Did your idea help?

> What I saw: 

---
## Step 4: One-hot encoding of a categorical feature

One-hot encoding turns a categorical feature (here equipment_id) into one column per category. In the training table, a row has a 1 in the column of its own group and a 0 in every other column, so each training row has exactly one 1. The validation group E5 is a machine the model never saw, so it gets a column of its own (`is_E5`) that the training table lacks. If we squeeze E5 into the four training columns, its row is all zeros. We only look at this here. We do not use equipment_id as a feature.

In [ ]:
one_hot_train = pd.get_dummies(train_df["equipment_id"], prefix="is").astype(int)
print(pd.concat([train_df["equipment_id"], one_hot_train], axis=1).head(5).to_string(index=False))

one_hot_valid = pd.get_dummies(valid_df["equipment_id"], prefix="is").astype(int)
print()
print("columns the model learned from:", list(one_hot_train.columns))
print("columns the validation group has:  ", list(one_hot_valid.columns))
print()
print("E5 rows squeezed into the training columns (all zeros):")
print(one_hot_valid.reindex(columns=one_hot_train.columns, fill_value=0).head(2).to_string(index=False))

**READ the output.** The validation group has a column (`is_E5`) that the training table does not, and E5 squeezed into the training columns is all zeros. What can a model say about a crusher group it has never seen from four zeros? Why do we leave `equipment_id` out of the features?

> My answer: 

---
## Step 5: Scaling and ridge regression

Ridge regression adds an L2 penalty on large weights. This is regularisation, and alpha is the regularisation strength. The penalty depends on the size of the weights, which depends on the units of the features, so we standardise first: scaled value (z-score) = (value minus the training mean) divided by the training standard deviation. `StandardScaler` is the scikit-learn tool that does this: it learns the mean and standard deviation from the training set only, then applies them to any rows.

In [ ]:
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge

X_train_imputed = SimpleImputer(strategy="median").fit_transform(X_train)   # missing values imputed with the training median
scaler = StandardScaler().fit(X_train_imputed)
print("training mean of load_pct:", round(scaler.mean_[0], 1), " standard deviation:", round(scaler.scale_[0], 1))

# part A: ridge regression on two features, with a few values of alpha
for alpha in [0.1, 1, 10]:
    ridge_model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=alpha))
    ridge_model.fit(X_train, y_train)
    mae = mean_absolute_error(y_valid, ridge_model.predict(X_valid))
    print(f"Ridge, two features, alpha {alpha:g}: MAE {mae:.2f} kWh/t")

**PREDICT first.** Linear regression on two features is not very flexible. Will a stronger penalty help it?

Write your guess here before you run the next cell:

> My prediction: 

**READ the output.** Does a bigger alpha lower the error of the linear model? When does ridge regression help, and when does it not?

> My answer: 

In [ ]:
# part B: a flexible curve (a sixth-degree polynomial of load_pct alone) with and without regularisation
load_train, load_valid = train_df[["load_pct"]], valid_df[["load_pct"]]

for name, last_step in [("degree 6, no penalty", LinearRegression()), ("degree 6, ridge alpha 1", Ridge(alpha=1)), ("degree 6, ridge alpha 100", Ridge(alpha=100))]:
    curve_model = make_pipeline(PolynomialFeatures(degree=6, include_bias=False), StandardScaler(), last_step)
    curve_model.fit(load_train, y_train)
    train_mae = mean_absolute_error(y_train, curve_model.predict(load_train))
    valid_mae = mean_absolute_error(y_valid, curve_model.predict(load_valid))
    biggest = abs(curve_model.steps[-1][1].coef_).max()       # the largest weight (coefficient) the model uses
    print(f"{name}: training MAE {train_mae:.2f}, validation MAE {valid_mae:.2f} kWh/t, biggest weight {biggest:.0f}")

**READ the output.** What happens to the biggest weight as the penalty grows? At which alpha is the validation error lowest? This curve uses load_pct alone, so it is a different model from the ridge regression on two features in part A. Compare these three rows only with each other.

> My answer: 

**The ridge regression loss.** Ridge regression makes this number small: loss = sum of squared errors + alpha times the sum of squared weights (the L2 penalty). The next cell works it out for linear regression on the two scaled features. Alpha 0 is ordinary linear regression.

In [ ]:
# part C: the two parts of the ridge regression loss, on the scaled training set
X_train_scaled = scaler.transform(X_train_imputed)
for alpha in [0, 0.1, 1, 10]:
    ridge_fit = Ridge(alpha=alpha).fit(X_train_scaled, y_train)
    residuals = np.asarray(y_train) - ridge_fit.predict(X_train_scaled)     # residual = actual minus predicted
    squared_errors = (residuals ** 2).sum()
    squared_weights = (ridge_fit.coef_ ** 2).sum()
    print(f"alpha {alpha:g}: weights {ridge_fit.coef_[0]:.2f} {ridge_fit.coef_[1]:.2f}, sum of squared errors {squared_errors:.1f}, alpha x sum of squared weights {alpha * squared_weights:.1f}, loss {squared_errors + alpha * squared_weights:.1f}")

**READ the output.** As alpha grows, what happens to the two weights, to the sum of squared errors and to the penalty part? Check one row by hand: do the two parts add up to the loss?

> My answer: 

**CHANGE one thing.** In part A, add 1000 to the list of alphas and run it again. How does the validation error change as alpha grows? In part B, at which alpha is the validation error lowest?

> What I saw: 

---
## Step 6: k-nearest neighbours (k-NN)

k-NN means k-nearest neighbours. To predict a new row, the model finds the k training rows closest to it, measured on the scaled features, and averages their targets. The distance between two rows is d = square root of ((gap in scaled load)^2 + (gap in scaled ambient)^2): square each gap, add the two, then take the square root (Euclidean distance). k is a hyperparameter: a setting you choose before training.

**PREDICT first.** Which k will give the smallest error: 1, 3 or 20?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

for k in [1, 3, 20]:
    knn_model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsRegressor(n_neighbors=k))
    knn_model.fit(X_train, y_train)
    mae = mean_absolute_error(y_valid, knn_model.predict(X_valid))
    print(f"k = {k}: MAE {mae:.2f} kWh/t")

**READ the output.** Why is k = 20 worse than k = 3? Why is k = 1 risky?

> My answer: 

In [ ]:
# open the box: which three training rows are closest to the first validation row?
X_valid_scaled = scaler.transform(SimpleImputer(strategy="median").fit(X_train).transform(X_valid))
knn_3 = KNeighborsRegressor(n_neighbors=3).fit(X_train_scaled, y_train)
distances, positions = knn_3.kneighbors(X_valid_scaled[:1])
neighbours = train_df.iloc[positions[0]]
print("new row: record", valid_df["record_id"].iloc[0], " actual", y_valid.iloc[0])
print(neighbours[["record_id", "load_pct", "ambient_c", "target"]].to_string(index=False))
print("distances to the 3 neighbours:", ", ".join(f"{d:.3f}" for d in distances[0]))
print(f"average of the 3 targets: {neighbours['target'].mean():.1f} kWh/t")

**CHANGE one thing.** Change `n_neighbors=3` to 5 in the last cell and run it again. Write down the three or five record numbers you see.

> What I saw: 

---
## Step 7: The model comparison table

Five candidate models, one loop: this is model comparison with the same protocol. Same training set, same validation set, same metrics, same seed. The cross-validated columns use GroupKFold cross-validation: each fold holds out one whole equipment group (E1, then E2, E3 and E4), and the model learns from the other three groups, so each fold plays a machine the model has never met. `CV mean` is the mean of the four fold MAEs. `CV lowest` and `CV highest` are the smallest and largest fold MAE.

**PREDICT first.** Which model will have the lowest cross-validated error? Will the same model win on the validation set?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score

models["Ridge regression, alpha 1"] = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=1))
models["k-NN, k = 3"] = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsRegressor(n_neighbors=3))

groups = train_df["equipment_id"]
cv = GroupKFold(n_splits=4)

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    cv_scores = -cross_val_score(model, X_train, y_train, cv=cv, groups=groups, scoring="neg_mean_absolute_error")
    mae = mean_absolute_error(y_valid, y_pred)
    rmse = mean_squared_error(y_valid, y_pred) ** 0.5
    rows.append([name, mae, rmse, cv_scores.mean(), cv_scores.min(), cv_scores.max()])
results = pd.DataFrame(rows, columns=["model", "valid MAE", "valid RMSE", "CV mean", "CV lowest", "CV highest"]).round(2)
print(results.to_string(index=False))

**READ the output.** Which model would you recommend, and why? Use the unit in your answer. If two models are close, prefer the simpler one (parsimony). Which one is simpler to explain?

> My answer: 

In [ ]:
plt.figure(figsize=(6.5, 3.8))
plt.barh(results["model"], results["CV mean"])
plt.xlabel("cross-validated mean absolute error (kWh/t), lower is better")
plt.gca().invert_yaxis()
plt.show()

**What this table cannot tell you.** The data are synthetic, which means invented for teaching. The validation set has only 15 rows. The held-out test set (E6) has not been used, and it stays held out until you have chosen your model. If you use it and then change your choice, it is no longer a fair test.

In [ ]:
print("The test set has", len(test_df), "rows. We have not used it.")

---
## Step 8: Your decision memo

Fill in each sentence in your own words, with numbers from your results table and their unit. Double-click the cell to edit it. One page is enough. In industry a related document is called a model card.

**Decision memo: predicting crusher energy demand**

1. **Question.** We wanted to know whether we can predict ... 

2. **Data.** We used ... training rows and ... validation rows. The data are ... 

3. **Models tried.** We tried ... 

4. **Result.** The best model on the validation set was ..., with a mean absolute error of ... kWh/t. The baseline model had ... kWh/t. In cross-validation the ... 

5. **Limits.** The numbers cannot tell us ... Three limits are ... 

6. **Recommendation.** I recommend ... because ... 

7. **Next step.** Before we rely on it, we should ... 

---
## Step 9: Save your work

Choose **Runtime, then Restart session and run all**. If every cell runs without an error, choose **File, then Save a copy in Drive**. Keep the notebook and your memo: the project uses these steps. The Engineering ML project is an individual, timed, auto-graded 3-hour sitting in Week 12 (proposed, lecturer to confirm), and a practice version, the rehearsal, is released right after Class 4.